# GAFIME Full Python API Reference & Cookbook

Bu notebook GAFIME’nin Python frontend API yüzeyini **kullanıcı**, **developer/debug**, **native backend** ve **Rust helper/orchestration** katmanlarıyla dökümante eder.

Amaç:

- tüm önemli class/function import yollarını göstermek,
- parametreleri ve davranışı açıklamak,
- örnek kullanım senaryoları vermek,
- CUDA/Rust/Polars/sklearn yoksa güvenli fallback mesajı üretmek,
- agent/maintainer smoke-test zemini sağlamak.

## Katmanlar

```text
Public user API
  gafime.GafimeEngine
  gafime.EngineConfig
  gafime.ComputeBudget
  gafime.GafimeStreamer
  gafime.generate_tutorial
  gafime.sklearn.GafimeSelector

Developer/debug API
  gafime.backends.resolve_backend
  gafime.metrics.MetricSuite
  gafime.planning.combinations.*
  gafime.utils.arrays.*
  gafime.utils.safety.*
  gafime.validation.*

Native/debug API
  gafime.backends.native_cuda_backend.NativeCudaBackend
  gafime.backends.fused_kernel.*
  gafime.backends.native_metal_backend.NativeMetalBackend
  gafime.backends.core_backend.CoreBackend

Rust helper/orchestration API
  gafime_cpu.OTSEncoder
  gafime_cpu.BatchScheduler
  gafime_cpu.AsyncPipeline
  gafime_cpu.ContiguousLayout / ContiguousBucket
  gafime_cpu.DataQualityAnalyzer / CacheAwareScheduler / SmartScheduler
```


## 0. Environment and safe imports


In [ ]:
import sys, os, platform, importlib, inspect, subprocess, json, math
from pathlib import Path
import numpy as np

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)

def try_import(name):
    try:
        mod = importlib.import_module(name)
        print(f"✅ {name} imported", getattr(mod, "__version__", ""))
        return mod
    except Exception as e:
        print(f"⚠️  {name} not available: {type(e).__name__}: {e}")
        return None

gafime = try_import("gafime")
sklearn = try_import("sklearn")
polars = try_import("polars")
gafime_cpu = try_import("gafime_cpu")

def show_signature(obj, name=None):
    name = name or getattr(obj, "__name__", repr(obj))
    try:
        print(f"{name}{inspect.signature(obj)}")
    except Exception as e:
        print(f"{name}: signature unavailable ({e})")

def short_doc(obj, n=10):
    doc = inspect.getdoc(obj) or ""
    return "\n".join(doc.splitlines()[:n])


## 1. Top-level package API

`import gafime` ile export edilen ana semboller:

| Symbol | Amaç | Tipik kullanım |
|---|---|---|
| `GafimeEngine` | Ana analiz motoru | `GafimeEngine(config).analyze(X, y)` |
| `EngineConfig` | Analiz ayarları | metric, backend, seed, repeats |
| `ComputeBudget` | Arama/VRAM bütçesi | max combo size, limit, VRAM |
| `GafimeStreamer` | CSV/Parquet streaming | büyük dosyayı chunk okumak |
| `generate_tutorial` | Tutorial notebook üretmek | `generate_tutorial('x.ipynb')` |
| `GafimeSelector` | sklearn transformer | Pipeline içine interaction feature eklemek |
| `__version__` | Paket sürümü | `gafime.__version__` |


In [ ]:
if gafime:
    print("gafime.__version__ =", getattr(gafime, "__version__", None))
    print("gafime.__all__ =", getattr(gafime, "__all__", None))
    for name in getattr(gafime, "__all__", []):
        obj = getattr(gafime, name, None)
        print("\n---", name, "---")
        print(obj)
        if obj is not None:
            show_signature(obj, name)
            doc = short_doc(obj, 4)
            if doc:
                print(doc)


## 2. `ComputeBudget`

`ComputeBudget` arama uzayını ve GPU/VRAM davranışını kontrol eder.

| Parametre | Varsayılan | Ne işe yarar? |
|---|---:|---|
| `max_comb_size` | `2` | Interaction derinliği. `1`: unary, `2`: pair, `3`: trio |
| `max_combinations_per_k` | `5000` | Her interaction derecesi için maksimum aday sayısı |
| `top_features_for_higher_k` | `50` | higher-order aramada en iyi unary feature shortlist |
| `max_generated_features` | `0` | gelecekte üretilen feature limiti için alan |
| `keep_in_vram` | `True` | GPU varsa veriyi VRAM'de tutmayı dene |
| `vram_budget_mb` | `6144` | GPU memory üst sınırı |

### Ne zaman değiştirirsin?

- Hızlı test: `max_comb_size=2`, `metric_names=('pearson',)`.
- Derin keşif: `max_comb_size=3`, `top_features_for_higher_k=20-50`.
- Laptop GPU: `vram_budget_mb` değerini konservatif tut.
- CPU debug: `EngineConfig(backend='cpu')`.


In [ ]:
from dataclasses import asdict, fields

if gafime:
    from gafime import ComputeBudget
    show_signature(ComputeBudget)
    b = ComputeBudget()
    print("Default ComputeBudget:")
    print(asdict(b))
    print("\nFields:")
    for f in fields(ComputeBudget):
        print(f"- {f.name}: default={f.default!r}, type={f.type}")

    conservative_budget = ComputeBudget(
        max_comb_size=2,
        max_combinations_per_k=10_000,
        top_features_for_higher_k=30,
        keep_in_vram=True,
        vram_budget_mb=4096,
    )
    print("\nCustom conservative budget:")
    print(asdict(conservative_budget))


## 3. `EngineConfig`

`EngineConfig`, `GafimeEngine` davranışını belirler.

| Parametre | Varsayılan | Açıklama |
|---|---:|---|
| `budget` | `ComputeBudget()` | Arama ve memory sınırları |
| `metric_names` | `('pearson','spearman','mutual_info','r2')` | Hangi metriklerle skorlanacak |
| `num_repeats` | `3` | Stability/bootstrap tekrar sayısı |
| `permutation_tests` | `25` | Random target permutation test sayısı |
| `random_seed` | `7` | Determinizm |
| `stability_std_threshold` | `0.10` | Karar için max metric std |
| `permutation_p_threshold` | `0.05` | Significance eşiği |
| `mi_bins` | `16` | Mutual information histogram bin sayısı |
| `backend` | `'auto'` | `auto`, `cuda`, `gpu`, `metal`, `cpu`, `numpy`, `core`, `cpp` |
| `device_id` | `0` | CUDA device index |


In [ ]:
if gafime:
    from gafime import EngineConfig, ComputeBudget
    show_signature(EngineConfig)
    print("Default EngineConfig:")
    print(asdict(EngineConfig()))

    fast_cfg = EngineConfig(
        metric_names=("pearson",),
        budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=50_000),
        permutation_tests=10,
        backend="auto",
        random_seed=42,
    )
    deep_cfg = EngineConfig(
        metric_names=("pearson", "spearman", "mutual_info"),
        budget=ComputeBudget(max_comb_size=3, top_features_for_higher_k=25),
        permutation_tests=100,
        permutation_p_threshold=0.01,
    )
    cpu_debug_cfg = EngineConfig(
        metric_names=("pearson", "r2"),
        backend="cpu",
        permutation_tests=5,
        num_repeats=2,
    )
    print("\nFast config:", asdict(fast_cfg))
    print("\nDeep config:", asdict(deep_cfg))
    print("\nCPU debug config:", asdict(cpu_debug_cfg))


## 4. Synthetic dataset helper


In [ ]:
def make_planted_interaction_dataset(
    n_samples=20_000,
    n_features=16,
    pair=(3, 7),
    noise=0.25,
    seed=42,
    dtype=np.float32,
):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n_samples, n_features)).astype(dtype)
    y = X[:, pair[0]] * X[:, pair[1]] + noise * rng.standard_normal(n_samples).astype(dtype)
    names = [f"f{i}" for i in range(n_features)]
    return X, y, names

X, y, feature_names = make_planted_interaction_dataset()
print(X.shape, X.dtype, y.shape, y.dtype)
print("Planted signal: f3 × f7")


## 5. `GafimeEngine`

### Signature

```python
GafimeEngine(config: EngineConfig | None = None)
```

### Ana metod

```python
report = engine.analyze(X, y, feature_names=None)
```

### `analyze` parametreleri

| Parametre | Tip | Açıklama |
|---|---|---|
| `X` | 2D array-like | `(n_samples, n_features)` numeric matrix |
| `y` | 1D array-like | `(n_samples,)` numeric target |
| `feature_names` | optional iterable[str] | kolon isimleri; verilmezse `f0`, `f1`, ... |

### Çalışma mantığı

1. `coerce_inputs`
2. `validate_budget`
3. `resolve_backend`
4. `plan_unary`
5. `select_top_features`
6. `plan_higher_order`
7. `backend.score_combos`
8. `StabilityAnalyzer`
9. `PermutationTester`
10. `DiagnosticReport`


In [ ]:
if gafime:
    from gafime import GafimeEngine, EngineConfig, ComputeBudget
    show_signature(GafimeEngine)
    show_signature(GafimeEngine.analyze, "GafimeEngine.analyze")

    cfg = EngineConfig(
        metric_names=("pearson",),
        budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=20_000),
        permutation_tests=10,
        num_repeats=3,
        backend="auto",
        random_seed=42,
    )
    engine = GafimeEngine(cfg)
    report = engine.analyze(X, y, feature_names=feature_names)
    print("Backend:", report.backend)
    print("Decision:", report.decision)
    print("Warnings:", report.warnings[:5])
    print("n interactions:", len(report.interactions))


## 6. Reading `DiagnosticReport`

| Alan | Tip | Açıklama |
|---|---|---|
| `config` | `EngineConfig` | çalıştırılan ayarlar |
| `feature_names` | list[str] | normalleştirilmiş feature adları |
| `interactions` | list[`InteractionResult`] | combo + metrik skorları |
| `stability` | list[`StabilityResult`] | bootstrap mean/std |
| `permutations` | list[`PermutationResult`] | null dağılım p-value |
| `warnings` | list[str] | memory/budget/backend uyarıları |
| `decision` | `Decision` | signal var/yok kararı |
| `backend` | `BackendInfo` | hangi backend çalıştı |
| `to_dict()` | method | dict çıktısı |


In [ ]:
if gafime:
    print("DiagnosticReport type:", type(report))
    print("Decision:", report.decision.signal_detected, "|", report.decision.message)
    print("Backend:", report.backend.name, report.backend.device, "GPU?", report.backend.is_gpu)

    print("\nTop 10 by |pearson|:")
    top = sorted(report.interactions, key=lambda r: abs(r.metrics.get("pearson", 0.0)), reverse=True)[:10]
    for r in top:
        print(f"{' × '.join(r.feature_names):<20} combo={r.combo} metrics={r.metrics}")

    print("\nFirst stability result:", report.stability[0] if report.stability else "No stability")
    print("First permutation result:", report.permutations[0] if report.permutations else "No permutation")
    print("report.to_dict keys:", report.to_dict().keys())


## 7. Report to DataFrame helper


In [ ]:
def report_to_dataframe(report):
    try:
        import pandas as pd
    except ImportError:
        print("pandas is not installed.")
        return None

    stability_map = {r.combo: r for r in report.stability}
    perm_map = {r.combo: r for r in report.permutations}
    rows = []
    for r in report.interactions:
        row = {"combo": r.combo, "feature_names": " × ".join(r.feature_names), "arity": len(r.combo)}
        for m, v in r.metrics.items():
            row[m] = v
            row[f"abs_{m}"] = abs(v) if m in ("pearson", "spearman") else v
            if r.combo in stability_map:
                row[f"{m}_mean"] = stability_map[r.combo].metrics_mean.get(m)
                row[f"{m}_std"] = stability_map[r.combo].metrics_std.get(m)
            if r.combo in perm_map:
                row[f"{m}_p"] = perm_map[r.combo].p_values.get(m)
        rows.append(row)
    return pd.DataFrame(rows)

if gafime:
    df_report = report_to_dataframe(report)
    if df_report is not None:
        display(df_report.sort_values("abs_pearson", ascending=False).head(10))


## 8. Metrics API

`MetricSuite` metrik isimlerini doğrular ve her combo vector'ü için skorları üretir.

Supported metrics:

```python
SUPPORTED_METRICS = ('pearson', 'spearman', 'mutual_info', 'r2')
```

Raw metric functions:

- `pearson_corr(x, y, xp=np)`
- `spearman_corr(x, y, xp=np)`
- `mutual_info(x, y, bins=16, xp=np)`
- `linear_r2(x, y, xp=np)`


In [ ]:
try:
    from gafime.metrics import MetricSuite, SUPPORTED_METRICS
    from gafime.metrics import cpu_metrics

    print("SUPPORTED_METRICS =", SUPPORTED_METRICS)
    show_signature(MetricSuite)
    show_signature(MetricSuite.score, "MetricSuite.score")

    for fn_name in ["pearson_corr", "spearman_corr", "mutual_info", "linear_r2"]:
        fn = getattr(cpu_metrics, fn_name)
        show_signature(fn, fn_name)

    x_vec = X[:, 3] * X[:, 7]
    suite = MetricSuite(("pearson", "spearman", "mutual_info", "r2"), mi_bins=16)
    print("\nMetricSuite score:", suite.score(x_vec, y))
except Exception as e:
    print("Metrics API demo failed:", type(e).__name__, e)


## 9. Planning API

Developer/debug API. `GafimeEngine` içeride bunları kullanır.

| Fonksiyon | Parametreler | Çıktı |
|---|---|---|
| `plan_unary(n_features, max_count, rng)` | feature sayısı, cap, RNG | unary combos + warnings |
| `select_top_features(feature_scores, top_n)` | `{feature_idx: score}` | top feature index listesi |
| `plan_higher_order(feature_indices, max_comb_size, max_combinations_per_k, rng)` | shortlist + k limitleri | pair/trio/... combos + warnings |
| `plan_combinations(n_features, budget, feature_scores, rng)` | birleşik planlayıcı | unary+higher combos + warnings |


In [ ]:
try:
    from gafime.planning.combinations import plan_unary, select_top_features, plan_higher_order, plan_combinations
    from gafime import ComputeBudget

    rng = np.random.default_rng(123)
    unary, warn = plan_unary(n_features=8, max_count=20, rng=rng)
    print("unary:", unary, "warnings:", warn)

    feature_scores = {i: float(i) / 10 for i in range(8)}
    top_features = select_top_features(feature_scores, top_n=4)
    print("top_features:", top_features)

    higher, warn = plan_higher_order(top_features, max_comb_size=3, max_combinations_per_k=10, rng=rng)
    print("higher combos:", higher, "warnings:", warn)

    combos, warn = plan_combinations(
        n_features=8,
        budget=ComputeBudget(max_comb_size=3, max_combinations_per_k=10, top_features_for_higher_k=4),
        feature_scores=feature_scores,
        rng=rng,
    )
    print("all planned combos:", combos)
except Exception as e:
    print("Planning API demo failed:", type(e).__name__, e)


## 10. Array and safety utilities

| Fonksiyon | Açıklama |
|---|---|
| `coerce_inputs(X, y, feature_names=None)` | X/y shape, dtype, finite check; contiguous float64 döndürür |
| `build_interaction_vector(X, combo, xp=np)` | unary veya centered-product interaction vector üretir |
| `estimate_combinations(n_features, k)` | `C(n_features, k)` |
| `validate_budget(n_features, budget)` | invalid budget için error/warning |
| `cap_combinations(count, limit)` | `min(count, limit)` |


In [ ]:
try:
    from gafime.utils.arrays import coerce_inputs, build_interaction_vector
    from gafime.utils.safety import estimate_combinations, validate_budget, cap_combinations
    from gafime import ComputeBudget

    X2, y2, names2 = coerce_inputs([[1,2],[3,4],[5,6]], [0.1, 0.2, 0.3], ["a", "b"])
    print("coerce_inputs:", X2, X2.dtype, y2, y2.dtype, names2, sep="\n")
    print("unary vector:", build_interaction_vector(X2, (0,)))
    print("pair centered product:", build_interaction_vector(X2, (0, 1)))
    print("C(100,2):", estimate_combinations(100, 2))
    print("budget warnings:", validate_budget(3, ComputeBudget(max_comb_size=4)))
    print("cap:", cap_combinations(1_000_000, 5_000))
except Exception as e:
    print("Utility API demo failed:", type(e).__name__, e)


## 11. Backend resolver and base backend API

`resolve_backend(config, X, y)` compute backend seçer.

Priority:

```text
CUDA native → Metal native → C++ core → NumPy fallback
```

Backend base methods:

- `info()`
- `check_budget(X, y, budget)`
- `to_device(array)`
- `to_host(array)`
- `build_interaction_vector(X, combo)`
- `score_combos(X, y, combos, metric_suite)`
- `sample_indices(n_samples, rng)`
- `permute(y, rng)`
- `estimate_bytes(X, y)`


In [ ]:
try:
    from gafime.backends import resolve_backend
    from gafime.backends.base import Backend
    from gafime import EngineConfig

    for requested in ["auto", "cuda", "metal", "core", "cpu", "numpy"]:
        try:
            cfg = EngineConfig(backend=requested, metric_names=("pearson",))
            backend, warnings = resolve_backend(cfg, X[:100], y[:100])
            info = backend.info()
            print(f"{requested:>6} -> {info.name:<18} device={info.device!r} gpu={info.is_gpu} warnings={warnings[:2]}")
        except Exception as e:
            print(f"{requested:>6} -> error: {type(e).__name__}: {e}")

    print("\nBackend base signatures:")
    for name in ["info", "check_budget", "to_device", "to_host", "build_interaction_vector", "score_combos", "sample_indices", "permute", "estimate_bytes"]:
        show_signature(getattr(Backend, name), f"Backend.{name}")
except Exception as e:
    print("Backend resolver demo failed:", type(e).__name__, e)


## 12. Direct validation API

Normalde `GafimeEngine.analyze` bunları içeride çağırır.

### `StabilityAnalyzer(metric_suite, backend).assess(...)`

Bootstrap sample'lar üzerinde aynı combo'ları skorlar; metric mean/std döndürür.

### `PermutationTester(metric_suite, backend).test(...)`

`y` değerini permute ederek null dağılım üretir; actual score'a göre empirical p-value hesaplar.


In [ ]:
try:
    from gafime.validation import StabilityAnalyzer, PermutationTester
    from gafime.metrics import MetricSuite
    from gafime.backends import resolve_backend
    from gafime import EngineConfig

    cfg = EngineConfig(metric_names=("pearson",), backend="auto")
    backend, warnings = resolve_backend(cfg, X, y)
    suite = MetricSuite(("pearson",))
    combos = [(3,), (7,), (3, 7), (0, 1)]
    rng = np.random.default_rng(42)

    actual_scores = backend.score_combos(X, y, combos, suite)
    print("actual_scores:", actual_scores)

    stability = StabilityAnalyzer(suite, backend).assess(X, y, combos, repeats=3, rng=rng)
    print("\nstability:")
    for r in stability:
        print(r)

    permutations = PermutationTester(suite, backend).test(X, y, combos, num_permutations=10, rng=rng, actual_scores=actual_scores)
    print("\npermutations:")
    for r in permutations:
        print(r)
except Exception as e:
    print("Validation API demo failed:", type(e).__name__, e)


## 13. `GafimeSelector`: scikit-learn transformer

```python
GafimeSelector(k=10, backend='auto', metric='pearson', operator='multiply', n_jobs=-1, verbose=False)
```

Çalışma mantığı:

1. `fit(X, y)` → GAFIME ile pairwise interaction skorlar.
2. En iyi `k` pair interaction'ı `top_interactions_` içine koyar.
3. `transform(X)` → orijinal X'e yeni interaction feature kolonları ekler.

Operator seçenekleri:

- `multiply`: `x_i * x_j`
- `add`: `x_i + x_j`
- `subtract`: `x_i - x_j`
- `divide`: `x_i / (x_j + 1e-8)`


In [ ]:
try:
    from gafime.sklearn import GafimeSelector

    show_signature(GafimeSelector)
    show_signature(GafimeSelector.fit, "GafimeSelector.fit")
    show_signature(GafimeSelector.transform, "GafimeSelector.transform")

    selector = GafimeSelector(k=5, backend="auto", metric="pearson", operator="multiply")
    selector.fit(X, y)
    X_aug = selector.transform(X[:10])

    print("top_interactions_:", selector.top_interactions_)
    print("n_features_in_:", selector.n_features_in_)
    print("before:", X[:10].shape, "after:", X_aug.shape)
except Exception as e:
    print("GafimeSelector demo failed:", type(e).__name__, e)


## 14. `GafimeSelector` in a sklearn Pipeline


In [ ]:
try:
    from gafime.sklearn import GafimeSelector
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    base_pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    gafime_pipe = Pipeline([
        ("gafime", GafimeSelector(k=5, backend="auto", metric="pearson", operator="multiply")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ])

    base_pipe.fit(X_train, y_train)
    gafime_pipe.fit(X_train, y_train)

    print("Base R2:", r2_score(y_test, base_pipe.predict(X_test)))
    print("GAFIME augmented R2:", r2_score(y_test, gafime_pipe.predict(X_test)))
    print("Selected pairs:", gafime_pipe.named_steps["gafime"].top_interactions_)
except Exception as e:
    print("sklearn pipeline demo failed:", type(e).__name__, e)


## 15. `GafimeStreamer`: VRAM-aware CSV/Parquet streaming

```python
GafimeStreamer(file_path, target_cols=None, y_col=None)
```

| API | Açıklama |
|---|---|
| `.total_rows` | lazy row count, cache'lenir |
| `.estimate_optimal_batch_size(vram_budget_gb=6.0, include_output=True, n_combos=256)` | VRAM'e göre row batch size |
| `.stream(batch_size=None, vram_budget_gb=6.0)` | `X_chunk` üretir |
| `.stream_with_target(batch_size=None, vram_budget_gb=6.0)` | `(X_chunk, y_chunk)` üretir |
| `create_streamer(...)` | convenience constructor |
| `benchmark_streaming(file_path, batch_size=None, n_batches=5)` | basit benchmark |


In [ ]:
try:
    from gafime import GafimeStreamer
    from gafime.io import create_streamer, benchmark_streaming
    show_signature(GafimeStreamer)
    show_signature(GafimeStreamer.estimate_optimal_batch_size, "GafimeStreamer.estimate_optimal_batch_size")
    show_signature(GafimeStreamer.stream, "GafimeStreamer.stream")
    show_signature(GafimeStreamer.stream_with_target, "GafimeStreamer.stream_with_target")
    show_signature(create_streamer)
    show_signature(benchmark_streaming)
except Exception as e:
    print("Streamer API import failed:", type(e).__name__, e)


In [ ]:
try:
    import tempfile
    import polars as pl
    from gafime import GafimeStreamer

    tmp_dir = Path(tempfile.mkdtemp())
    path = tmp_dir / "gafime_stream_demo.parquet"

    demo = {**{f"f{i}": X[:5000, i].astype(np.float32) for i in range(min(8, X.shape[1]))}, "target": y[:5000].astype(np.float32)}
    pl.DataFrame(demo).write_parquet(path)

    streamer = GafimeStreamer(path, y_col="target")
    print("total_rows:", streamer.total_rows)
    print("n_features:", streamer.n_features)
    print("optimal batch:", streamer.estimate_optimal_batch_size(vram_budget_gb=1.0, n_combos=64))

    for i, (X_chunk, y_chunk) in enumerate(streamer.stream_with_target(batch_size=1500)):
        print(f"chunk {i}: X={X_chunk.shape} {X_chunk.dtype}, y={y_chunk.shape} {y_chunk.dtype}")
        if i >= 2:
            break
except Exception as e:
    print("Streaming demo skipped/failed:", type(e).__name__, e)


## 16. Tutorial generator and CLI

Programatik tutorial:

```python
from gafime import generate_tutorial
generate_tutorial('gafime_tutorial.ipynb')
```

CLI:

```bash
gafime --version
gafime --check
gafime --init
gafime --init -o custom.ipynb
```


In [ ]:
try:
    from gafime import generate_tutorial
    show_signature(generate_tutorial)
    print(short_doc(generate_tutorial, 20))

    # Uncomment to write a tutorial file:
    # generate_tutorial("generated_gafime_tutorial.ipynb")

    try:
        result = subprocess.run([sys.executable, "-m", "gafime.cli", "--help"], capture_output=True, text=True, timeout=10)
        print("\nCLI help:")
        print(result.stdout or result.stderr)
    except Exception as e:
        print("CLI subprocess demo skipped:", e)
except Exception as e:
    print("Tutorial/CLI demo failed:", type(e).__name__, e)


## 17. Native CUDA frontend/debug API: `NativeCudaBackend`

Normal kullanıcı doğrudan bunu çağırmaz; `resolve_backend`/`GafimeEngine` seçer.

Önemli methodlar:

| Method | Açıklama |
|---|---|
| `NativeCudaBackend(device_id=0)` | native CUDA library yükler, CUDA availability kontrol eder |
| `.info()` | GPU name, compute capability, memory info |
| `.check_budget(X, y, budget)` | VRAM budget uygun mu? |
| `.build_interaction_vector(X, combo)` | legacy CUDA vector üretimi |
| `.score_combos(X, y, combos, metric_suite)` | fast bucket path veya legacy path |

Fast path koşulu:

- `_has_bucket_api == True`
- metric suite sadece `('pearson',)`
- combo uzunlukları `1` veya `2`


In [ ]:
try:
    from gafime.backends.native_cuda_backend import NativeCudaBackend
    from gafime.metrics import MetricSuite

    show_signature(NativeCudaBackend)
    backend = NativeCudaBackend(device_id=0)
    print("CUDA backend info:", backend.info())
    print("_has_bucket_api:", getattr(backend, "_has_bucket_api", None))

    suite = MetricSuite(("pearson",))
    scores = backend.score_combos(X, y, [(3,), (7,), (3, 7), (0, 1)], suite)
    print("scores:", scores)
except Exception as e:
    print("NativeCudaBackend demo skipped/failed:", type(e).__name__, e)


## 18. Native C++ Core backend: `CoreBackend`


In [ ]:
try:
    from gafime.backends.core_backend import CoreBackend
    from gafime.metrics import MetricSuite

    show_signature(CoreBackend)
    core = CoreBackend()
    print("Core info:", core.info())
    suite = MetricSuite(("pearson", "r2"))
    print(core.score_combos(X, y, [(3,7), (0,1)], suite))
except Exception as e:
    print("CoreBackend demo skipped/failed:", type(e).__name__, e)


## 19. Native Metal backend: `NativeMetalBackend`


In [ ]:
try:
    from gafime.backends.native_metal_backend import NativeMetalBackend
    show_signature(NativeMetalBackend)
    metal = NativeMetalBackend()
    print(metal.info())
except Exception as e:
    print("NativeMetalBackend demo skipped/failed:", type(e).__name__, e)


## 20. Fused kernel debug API: `gafime.backends.fused_kernel`

Low-level/debug API.

| Symbol | Açıklama |
|---|---|
| `get_gpu_config()` | CUDA auto-tuned GPU config |
| `GpuConfig` | gpu_name, block_size, max_blocks, sm_count, compute capability, L2 |
| `UnaryOp` | identity/log/exp/sqrt/tanh/sigmoid/square/... enum mirror |
| `InteractionType` | mult/add/sub/div/max/min enum mirror |
| `pearson_from_stats(...)` | sufficient stats → Pearson r |
| `unpack_stats(stats)` | 12-float stats array'i train/val dict'e böler |
| `compute_pearson_from_stats(stats)` | train/val Pearson döndürür |
| `FusedKernelWrapper` | per-call fused CUDA interaction wrapper |
| `StaticBucket` | preallocated VRAM bucket wrapper |


In [ ]:
try:
    from gafime.backends import fused_kernel as fk

    for name in ["get_gpu_config", "pearson_from_stats", "unpack_stats", "compute_pearson_from_stats", "FusedKernelWrapper", "StaticBucket"]:
        obj = getattr(fk, name, None)
        print("\n---", name, "---")
        print(obj)
        if obj:
            show_signature(obj, name)

    print("\nUnaryOp constants:")
    for k, v in fk.UnaryOp._names.items():
        print(k, v)

    print("\nInteractionType constants:")
    for k, v in fk.InteractionType._names.items():
        print(k, v)

    toy_stats = np.zeros(12, dtype=np.float32)
    toy_stats[0] = 5; toy_stats[1] = 15; toy_stats[2] = 30; toy_stats[3] = 55; toy_stats[4] = 220; toy_stats[5] = 110
    print("\nTrain/val stats:", fk.unpack_stats(toy_stats))
    print("Pearson:", fk.compute_pearson_from_stats(toy_stats))
except Exception as e:
    print("fused_kernel API demo failed:", type(e).__name__, e)


## 21. `FusedKernelWrapper` example

CUDA backend varsa çalışır.

```python
wrapper = FusedKernelWrapper()
stats = wrapper.compute(
    features=[X[:,0], X[:,1]],
    target=y,
    mask=fold_mask,
    ops=[UnaryOp.LOG, UnaryOp.SQRT],
    interaction=InteractionType.MULT,
    val_fold=0,
)
train_r, val_r = compute_pearson_from_stats(stats)
```


In [ ]:
try:
    from gafime.backends.fused_kernel import FusedKernelWrapper, UnaryOp, InteractionType, compute_pearson_from_stats

    mask = np.zeros(X.shape[0], dtype=np.uint8)
    wrapper = FusedKernelWrapper()
    print("CUDA available:", wrapper.cuda_available())

    stats = wrapper.compute(
        features=[X[:, 3], X[:, 7]],
        target=y,
        mask=mask,
        ops=[UnaryOp.IDENTITY, UnaryOp.IDENTITY],
        interaction=InteractionType.MULT,
        val_fold=255,
    )
    print("stats:", stats)
    print("pearson:", compute_pearson_from_stats(stats))
except Exception as e:
    print("FusedKernelWrapper demo skipped/failed:", type(e).__name__, e)


## 22. `StaticBucket`

Zero-malloc hot loop için tasarlanmıştır.

Önemli methodlar:

- `upload_feature(feature_idx, data)`
- `upload_target(target)`
- `upload_mask(mask)`
- `upload_all(features, target, mask)`
- `compute(feature_indices, ops, interaction_types=None, val_fold=0)`
- `compute_batch(feature_pairs, op_pairs, interactions, val_fold=0)`
- `interleaved_compute(...)`


In [ ]:
try:
    from gafime.backends.fused_kernel import StaticBucket, UnaryOp, InteractionType, compute_pearson_from_stats

    bucket = StaticBucket(n_samples=X.shape[0], n_features=5)
    features = [X[:, i] for i in range(5)]
    mask = np.zeros(X.shape[0], dtype=np.uint8)
    bucket.upload_all(features=features, target=y, mask=mask)

    stats = bucket.compute(
        feature_indices=[0, 1],
        ops=[UnaryOp.IDENTITY, UnaryOp.IDENTITY],
        interaction_types=[InteractionType.MULT],
        val_fold=255,
    )
    print("single stats:", stats)
    print("single pearson:", compute_pearson_from_stats(stats))

    stats_batch = bucket.compute_batch(
        feature_pairs=[(0, 1), (0, 2), (1, 2)],
        op_pairs=[(UnaryOp.IDENTITY, UnaryOp.IDENTITY)] * 3,
        interactions=[InteractionType.MULT] * 3,
        val_fold=255,
    )
    print("batch stats shape:", stats_batch.shape)
    del bucket
except Exception as e:
    print("StaticBucket demo skipped/failed:", type(e).__name__, e)


## 23. Rust helper/orchestration API: `gafime_cpu`

Rust modülü compute backend değildir; CPU helper/orchestration katmanıdır.

Beklenen sınıflar:

- `OTSEncoder`
- `BatchScheduler`
- `AsyncPipeline`
- `CacheAwareScheduler`
- `DataQualityAnalyzer`
- `ContiguousLayout`
- `ContiguousBucket`
- `SmartScheduler`

Not: `Xoshiro256PlusPlus` RNG şu an general RNG API olarak export edilmemiştir; `OTSEncoder` içinde private permutation RNG olarak kullanılır.


In [ ]:
try:
    import gafime_cpu
    print("gafime_cpu module:", gafime_cpu)
    print("version:", getattr(gafime_cpu, "__version__", None))

    for name in ["OTSEncoder", "BatchScheduler", "AsyncPipeline", "CacheAwareScheduler", "DataQualityAnalyzer", "ContiguousLayout", "ContiguousBucket", "SmartScheduler"]:
        obj = getattr(gafime_cpu, name, None)
        print(f"{name:24s} -> {obj}")
        if obj is not None:
            try:
                show_signature(obj, name)
            except Exception:
                pass
except Exception as e:
    print("gafime_cpu not available:", type(e).__name__, e)


## 24. Rust `OTSEncoder`: leakage-safe target encoding

```python
OTSEncoder(prior=0.5, prior_weight=1.0, seed=42, n_permutations=4)
```

Methods:

- `fit_transform(categories, targets)`
- `transform(categories)`
- `n_categories()`
- `get_prior()`

Çalışma mantığı:

- Random permutation üretir.
- Her category için yalnız önceki örneklerin target istatistiğini kullanır.
- Birden çok permutation varsa Rayon ile paralel hesaplayıp ortalamasını alır.
- RNG: internal `Xoshiro256PlusPlus`.


In [ ]:
try:
    from gafime_cpu import OTSEncoder

    categories = [0, 0, 1, 1, 2, 2, 2, 3, 3, 0]
    targets =    [1, 0, 1, 1, 0, 0, 1, 1, 0, 1]

    enc = OTSEncoder(prior=0.5, prior_weight=1.0, seed=42, n_permutations=4)
    encoded = enc.fit_transform(categories, [float(t) for t in targets])

    print("encoded:", encoded)
    print("n_categories:", enc.n_categories())
    print("prior:", enc.get_prior())
    print("transform seen/unseen:", enc.transform([0, 1, 2, 3, 999]))
except Exception as e:
    print("OTSEncoder demo skipped/failed:", type(e).__name__, e)


## 25. Rust `BatchScheduler`

BatchScheduler compute yapmaz; interaction batch'lerini GPU pipeline/compute için flat array formatına çevirir.

```python
BatchScheduler(max_blocks=96, cuda_dll_path=None)
```

Methods:

- `get_optimal_batch_sizes()`
- `optimal_batch_size()`
- `max_blocks()`
- `create_batches(feature_pairs, op_pairs, interactions)`
- `generate_all_pairs(n_features, ops, interaction_type)`

Output format:

```text
(indices, ops, interact, size)
indices:  [N * 2]
ops:      [N * 2]
interact: [N]
size:     N
```


In [ ]:
try:
    from gafime_cpu import BatchScheduler

    scheduler = BatchScheduler(max_blocks=96)
    print("optimal sizes:", scheduler.get_optimal_batch_sizes())
    print("optimal batch:", scheduler.optimal_batch_size())
    print("max blocks:", scheduler.max_blocks())

    feature_pairs = [(0, 1), (0, 2), (1, 2), (2, 3)]
    op_pairs = [(0, 0)] * len(feature_pairs)
    interactions = [0] * len(feature_pairs)

    batches = scheduler.create_batches(feature_pairs, op_pairs, interactions)
    print("\nmanual batches:")
    for b in batches:
        print(b)

    generated = scheduler.generate_all_pairs(n_features=4, ops=[0, 6], interaction_type=0)
    print("\ngenerated all-pair batches:")
    for b in generated[:3]:
        print(b)
except Exception as e:
    print("BatchScheduler demo skipped/failed:", type(e).__name__, e)


## 26. Rust `ContiguousLayout` and `ContiguousBucket`

### `ContiguousLayout`

- `ContiguousLayout(n_samples, n_features)`
- `.add_feature(feature)`
- `.set_target(target)`
- `.set_mask(mask)`
- `.is_complete()`
- `.info()`

Layout:

```text
[Feature0][Feature1]...[FeatureN][Target]
```

### `ContiguousBucket`

- `ContiguousBucket(layout)` → allocate + upload
- `.compute(feature_a, feature_b, op_a, op_b, interact_type, val_fold_id)`
- `.compute_batch(...)`
- `.close()`


In [ ]:
try:
    from gafime_cpu import ContiguousLayout, ContiguousBucket

    X_small = X[:1000, :4].astype(np.float32)
    y_small = y[:1000].astype(np.float32)

    layout = ContiguousLayout(n_samples=X_small.shape[0], n_features=X_small.shape[1])
    for j in range(X_small.shape[1]):
        layout.add_feature(X_small[:, j].tolist())
    layout.set_target(y_small.tolist())
    layout.set_mask([0] * X_small.shape[0])

    print("complete?", layout.is_complete())
    print("info:", layout.info())

    bucket = ContiguousBucket(layout)
    stats = bucket.compute(0, 1, 0, 0, 0, 255)
    print("stats len:", len(stats))
    print("stats:", stats)
    bucket.close()
except Exception as e:
    print("ContiguousLayout/Bucket demo skipped/failed:", type(e).__name__, e)


## 27. Rust `AsyncPipeline`

```python
AsyncPipeline(bucket_ptr: int, val_fold_id: int)
```

Methods:

- `.submit(indices, ops, interact)`
- `.pending()`
- `.wait()`
- `.process_all(batches)`
- `.close()`

Bu API bir `GafimeBucket` pointer ister. Genelde low-level `StaticBucket._bucket` gibi handle üzerinden kullanılır.


In [ ]:
try:
    from gafime_cpu import AsyncPipeline
    from gafime.backends.fused_kernel import StaticBucket

    X_small = X[:2000, :5]
    y_small = y[:2000]
    mask = np.zeros(X_small.shape[0], dtype=np.uint8)

    bucket = StaticBucket(n_samples=X_small.shape[0], n_features=5)
    bucket.upload_all([X_small[:, i] for i in range(5)], target=y_small, mask=mask)

    pipe = AsyncPipeline(bucket._bucket.value, 255)
    slot = pipe.submit([0, 1, 0, 2], [0, 0, 0, 0], [0, 0])
    print("submitted slot:", slot)
    print("pending:", pipe.pending())

    stats, batch_size = pipe.wait()
    print("batch_size:", batch_size)
    print("stats len:", len(stats))

    pipe.close()
    del bucket
except Exception as e:
    print("AsyncPipeline demo skipped/failed:", type(e).__name__, e)


## 28. Data quality / cache / smart scheduler introspection


In [ ]:
try:
    import gafime_cpu
    for cls_name in ["DataQualityAnalyzer", "CacheAwareScheduler", "SmartScheduler"]:
        cls = getattr(gafime_cpu, cls_name, None)
        print("\n==", cls_name, "==")
        print(cls)
        if cls is None:
            continue
        public = [x for x in dir(cls) if not x.startswith("_")]
        print("public attrs/methods:", public)
        try:
            show_signature(cls, cls_name)
        except Exception:
            pass
except Exception as e:
    print("Rust scheduler/data-quality introspection skipped:", type(e).__name__, e)


## 29. General RNG API status

Rust tarafında hızlı RNG var ama public Python general RNG API görünmüyor.

```text
Internal RNG: rand_xoshiro::Xoshiro256PlusPlus
Where used: OTSEncoder.random_permutation
Python exposed FastRng/XoshiroRng: not currently visible
Fallback: NumPy default_rng for frontend planning/stability/permutation
```

Proposed future API:

```python
from gafime_cpu import FastRng
rng = FastRng(seed=42)
perm = rng.permutation(100_000)
ints = rng.integers(0, 100, 10_000)
floats = rng.uniform(10_000)
```


In [ ]:
try:
    import gafime_cpu
    for name in ["FastRng", "XoshiroRng", "RandomGenerator", "Rng", "RNG"]:
        print(name, "->", getattr(gafime_cpu, name, None))
except Exception as e:
    print("gafime_cpu not available:", type(e).__name__, e)

rng = np.random.default_rng(42)
print("NumPy permutation sample:", rng.permutation(10))


## 30. Scenario: fast pair discovery


In [ ]:
if gafime:
    from gafime import GafimeEngine, EngineConfig, ComputeBudget

    cfg = EngineConfig(
        metric_names=("pearson",),
        budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=100_000),
        backend="auto",
        permutation_tests=10,
        num_repeats=3,
        random_seed=42,
    )
    report_fast = GafimeEngine(cfg).analyze(X, y, feature_names=feature_names)
    df_fast = report_to_dataframe(report_fast)
    print("backend:", report_fast.backend)
    print("decision:", report_fast.decision)
    if df_fast is not None:
        display(df_fast.query("arity == 2").sort_values("abs_pearson", ascending=False).head(10))


## 31. Scenario: robust evidence with stability + permutations


In [ ]:
if gafime:
    from gafime import GafimeEngine, EngineConfig, ComputeBudget

    cfg = EngineConfig(
        metric_names=("pearson", "spearman"),
        budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=20_000),
        num_repeats=5,
        permutation_tests=50,
        stability_std_threshold=0.05,
        permutation_p_threshold=0.05,
        random_seed=123,
    )
    report_robust = GafimeEngine(cfg).analyze(X, y, feature_names=feature_names)
    df_robust = report_to_dataframe(report_robust)
    print("decision:", report_robust.decision)
    if df_robust is not None:
        cols = [c for c in ["feature_names", "pearson", "pearson_std", "pearson_p", "spearman", "spearman_std", "spearman_p"] if c in df_robust.columns]
        display(df_robust.sort_values("abs_pearson", ascending=False)[cols].head(10))


## 32. Scenario: sklearn feature generation


In [ ]:
try:
    from gafime.sklearn import GafimeSelector
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge
    from sklearn.model_selection import cross_val_score

    pipe = Pipeline([
        ("gafime", GafimeSelector(k=8, backend="auto", metric="pearson", operator="multiply")),
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ])
    scores = cross_val_score(pipe, X, y, cv=3, scoring="r2")
    print("R2 CV scores:", scores)
    print("mean:", scores.mean())
except Exception as e:
    print("ML feature generation scenario skipped/failed:", type(e).__name__, e)


## 33. Scenario: categorical target encoding before GAFIME


In [ ]:
try:
    from gafime_cpu import OTSEncoder
    from gafime import GafimeEngine, EngineConfig, ComputeBudget

    rng = np.random.default_rng(42)
    n = 10_000
    cat_a = rng.integers(0, 50, size=n, dtype=np.uint32)
    cat_b = rng.integers(0, 20, size=n, dtype=np.uint32)
    num = rng.normal(size=n).astype(np.float32)
    y_cat = ((cat_a % 7) / 7.0 + (cat_b % 5) / 5.0 + 0.3 * num + rng.normal(0, 0.1, size=n)).astype(np.float32)

    enc_a = OTSEncoder(prior=float(np.mean(y_cat)), prior_weight=5.0, seed=42, n_permutations=4)
    enc_b = OTSEncoder(prior=float(np.mean(y_cat)), prior_weight=5.0, seed=123, n_permutations=4)

    a_ots = np.array(enc_a.fit_transform(cat_a.tolist(), y_cat.tolist()), dtype=np.float32)
    b_ots = np.array(enc_b.fit_transform(cat_b.tolist(), y_cat.tolist()), dtype=np.float32)

    X_cat = np.column_stack([a_ots, b_ots, num]).astype(np.float32)
    names = ["cat_a_ots", "cat_b_ots", "num"]

    report_cat = GafimeEngine(EngineConfig(
        metric_names=("pearson",),
        budget=ComputeBudget(max_comb_size=2),
        permutation_tests=10,
    )).analyze(X_cat, y_cat, feature_names=names)

    for r in sorted(report_cat.interactions, key=lambda r: abs(r.metrics.get("pearson", 0)), reverse=True)[:10]:
        print(r.feature_names, r.metrics)
except Exception as e:
    print("Categorical OTS + GAFIME scenario skipped/failed:", type(e).__name__, e)


## 34. Scenario: large file streaming


In [ ]:
try:
    import tempfile
    import polars as pl
    from gafime import GafimeStreamer, GafimeEngine, EngineConfig, ComputeBudget

    tmp = Path(tempfile.mkdtemp())
    path = tmp / "largeish_demo.parquet"

    n = 30_000
    rng = np.random.default_rng(0)
    data = {f"f{i}": rng.normal(size=n).astype(np.float32) for i in range(10)}
    data["target"] = (data["f2"] * data["f4"] + 0.2 * rng.normal(size=n)).astype(np.float32)
    pl.DataFrame(data).write_parquet(path)

    streamer = GafimeStreamer(path, y_col="target")
    engine = GafimeEngine(EngineConfig(
        metric_names=("pearson",),
        budget=ComputeBudget(max_comb_size=2, max_combinations_per_k=5000),
        permutation_tests=5,
    ))

    for batch_idx, (Xb, yb) in enumerate(streamer.stream_with_target(batch_size=10_000)):
        print("batch", batch_idx, Xb.shape, yb.shape)
        rb = engine.analyze(Xb, yb, feature_names=[f"f{i}" for i in range(10)])
        top = sorted(rb.interactions, key=lambda r: abs(r.metrics.get("pearson", 0)), reverse=True)[:3]
        for r in top:
            print(" ", r.feature_names, r.metrics)
        if batch_idx >= 1:
            break
except Exception as e:
    print("Large-file streaming scenario skipped/failed:", type(e).__name__, e)


## 35. API availability matrix


In [ ]:
checks = []

def check_symbol(import_path, symbol=None):
    try:
        mod = importlib.import_module(import_path)
        obj = getattr(mod, symbol) if symbol else mod
        checks.append((import_path + (f".{symbol}" if symbol else ""), True, type(obj).__name__, ""))
    except Exception as e:
        checks.append((import_path + (f".{symbol}" if symbol else ""), False, "", f"{type(e).__name__}: {e}"))

targets = [
    ("gafime", "GafimeEngine"), ("gafime", "EngineConfig"), ("gafime", "ComputeBudget"),
    ("gafime", "GafimeStreamer"), ("gafime", "generate_tutorial"),
    ("gafime.sklearn", "GafimeSelector"),
    ("gafime.backends", "resolve_backend"),
    ("gafime.backends.base", "Backend"), ("gafime.backends.base", "BackendInfo"),
    ("gafime.backends.core_backend", "CoreBackend"),
    ("gafime.backends.native_cuda_backend", "NativeCudaBackend"),
    ("gafime.backends.native_metal_backend", "NativeMetalBackend"),
    ("gafime.backends.fused_kernel", "FusedKernelWrapper"),
    ("gafime.backends.fused_kernel", "StaticBucket"),
    ("gafime.backends.fused_kernel", "UnaryOp"),
    ("gafime.backends.fused_kernel", "InteractionType"),
    ("gafime.metrics", "MetricSuite"), ("gafime.metrics", "SUPPORTED_METRICS"),
    ("gafime.validation", "StabilityAnalyzer"), ("gafime.validation", "PermutationTester"),
    ("gafime.planning.combinations", "plan_unary"),
    ("gafime.planning.combinations", "plan_higher_order"),
    ("gafime.utils.arrays", "coerce_inputs"), ("gafime.utils.arrays", "build_interaction_vector"),
    ("gafime.utils.safety", "estimate_combinations"),
    ("gafime_cpu", "OTSEncoder"), ("gafime_cpu", "BatchScheduler"), ("gafime_cpu", "AsyncPipeline"),
    ("gafime_cpu", "ContiguousLayout"), ("gafime_cpu", "ContiguousBucket"),
    ("gafime_cpu", "DataQualityAnalyzer"), ("gafime_cpu", "CacheAwareScheduler"), ("gafime_cpu", "SmartScheduler"),
]

for mod, sym in targets:
    check_symbol(mod, sym)

try:
    import pandas as pd
    df_checks = pd.DataFrame(checks, columns=["symbol", "available", "type", "error"])
    display(df_checks)
except Exception:
    for row in checks:
        print(row)


## 36. Common failure modes and fixes

| Belirti | Muhtemel sebep | Çözüm |
|---|---|---|
| `Native CUDA library not found` | CUDA wheel/lib yok veya path yanlış | wheel/local build/DLL path kontrolü |
| `CUDA not available` | GPU/driver/toolkit yok | CPU/Core backend kullan veya CUDA setup |
| `gafime_core ... missing` | C++ pybind extension build edilmemiş | `python setup.py build_ext --inplace` |
| `gafime_cpu not available` | Rust PyO3 extension build edilmemiş | cargo/rust build path kontrolü |
| `Polars is required` | streaming dependency yok | `pip install polars` |
| `sklearn` import hatası | sklearn optional dependency yok | `pip install scikit-learn` |
| `X and y must be finite` | NaN/Inf var | preprocessing/data quality check |
| Tensor backend çalışmıyor | intentional dummy/future endpoint | bug olarak takip etme; Torch/autograd entegrasyonu bekliyor |


## 37. Maintainer checklist

Yeni release veya agent-run öncesi:

- [ ] `gafime --check` çalışıyor mu?
- [ ] `GafimeEngine().analyze` synthetic planted interaction'ı buluyor mu?
- [ ] CUDA varsa `NativeCudaBackend` import + `.info()` çalışıyor mu?
- [ ] `metric_names=('pearson',)` + pairs fast path çalışıyor mu?
- [ ] CPU fallback/CoreBackend çalışıyor mu?
- [ ] `GafimeSelector.fit/transform` shape artırıyor mu?
- [ ] `GafimeStreamer` CSV/Parquet okuyabiliyor mu?
- [ ] `gafime_cpu.OTSEncoder` çalışıyor mu?
- [ ] `BatchScheduler.create_batches` doğru flat format üretiyor mu?
- [ ] Tensor dummy endpoint dokümantasyonda future seam olarak işaretli mi?


## 38. v0.4.0 discrete EngineConfig and ComputeBudget controls

GAFIME v0.4.0 adds discrete function search inside the existing `GafimeEngine` path. The feature is opt-in through `EngineConfig.enable_discrete_functions`; candidate volume stays under the existing budget object.

| Field | Layer | Default | Purpose |
|---|---|---:|---|
| `enable_discrete_functions` | `EngineConfig` | `False` | Enable threshold/interval/rectangle candidate families |
| `discrete_mode` | `EngineConfig` | `'soft'` | `'soft'` on all backends; `'hard'` only on CPU/NumPy |
| `discrete_ranking` | `EngineConfig` | `'split_aware'` | Rank discrete candidates with split-aware selector scores |
| `discrete_threshold_source` | `EngineConfig` | `'quantile'` | v0.4.0 fixed engine-generated thresholds |
| `discrete_gate_sharpness` | `EngineConfig` | `12.0` | Sigmoid/soft-gate steepness |
| `discrete_quantiles` | `EngineConfig` | `(0.05, ..., 0.95)` | Candidate threshold quantiles |
| `max_discrete_candidates` | `ComputeBudget` | `100_000` | Global cap for discrete candidates |
| `max_thresholds_per_feature` | `ComputeBudget` | `9` | Quantile threshold cap per feature |
| `max_intervals_per_feature` | `ComputeBudget` | `12` | Interval candidate cap per feature |
| `max_feature_pairs_for_rectangles` | `ComputeBudget` | `500` | Rectangle pair cap |
| `top_k_features_for_discrete` | `ComputeBudget` | `50` | Feature shortlist for discrete generation |


In [ ]:
try:
    from dataclasses import asdict, fields
    from gafime import ComputeBudget, EngineConfig
    from gafime.config import DEFAULT_DISCRETE_QUANTILES

    show_signature(EngineConfig)
    show_signature(ComputeBudget)
    print("DEFAULT_DISCRETE_QUANTILES =", DEFAULT_DISCRETE_QUANTILES)

    discrete_cfg = EngineConfig(
        enable_discrete_functions=True,
        discrete_mode="soft",
        discrete_ranking="split_aware",
        discrete_threshold_source="quantile",
        discrete_gate_sharpness=12.0,
        budget=ComputeBudget(
            max_discrete_candidates=100_000,
            max_thresholds_per_feature=9,
            max_intervals_per_feature=12,
            max_feature_pairs_for_rectangles=500,
            top_k_features_for_discrete=50,
        ),
    )
    print("\nDiscrete config:")
    print(asdict(discrete_cfg))

    print("\nEngineConfig discrete fields:")
    for f in fields(EngineConfig):
        if f.name.startswith("discrete") or f.name == "enable_discrete_functions":
            print(f"- {f.name}: default={f.default!r}, type={f.type}")

    print("\nComputeBudget discrete fields:")
    for f in fields(ComputeBudget):
        if "discrete" in f.name or "threshold" in f.name or "interval" in f.name or "rectangle" in f.name:
            print(f"- {f.name}: default={f.default!r}, type={f.type}")
except Exception as e:
    print("Discrete config introspection failed:", type(e).__name__, e)


## 39. Discrete function representation API

The discrete family is available in `gafime.discrete` for developer/debug use. Normal users usually enable it through `GafimeEngine`; direct helpers are useful for smoke tests, backend validation, and explaining candidates from a report.

Implemented families:

- `discrete_function_soft_threshold`
- `discrete_function_soft_interval`
- `discrete_function_value_gated_threshold`
- `discrete_function_soft_rectangle`
- `discrete_function_value_in_soft_rectangle`

Important helper types/functions:

- `DiscreteFunctionCandidate`
- `evaluate_discrete_candidate`
- `evaluate_discrete_mask`
- `score_discrete_candidates`
- `discrete_candidate_from_result`
- `describe_discrete_candidate`
- `discrete_feature_names`


In [ ]:
try:
    from gafime.discrete import (
        DiscreteFunctionCandidate,
        describe_discrete_candidate,
        discrete_feature_names,
        discrete_function_soft_interval,
        discrete_function_soft_rectangle,
        discrete_function_soft_threshold,
        discrete_function_value_gated_threshold,
        discrete_function_value_in_soft_rectangle,
        evaluate_discrete_candidate,
        evaluate_discrete_mask,
    )

    for obj in [
        DiscreteFunctionCandidate,
        discrete_function_soft_threshold,
        discrete_function_soft_interval,
        discrete_function_value_gated_threshold,
        discrete_function_soft_rectangle,
        discrete_function_value_in_soft_rectangle,
        evaluate_discrete_candidate,
        evaluate_discrete_mask,
        describe_discrete_candidate,
        discrete_feature_names,
    ]:
        show_signature(obj)

    if "X" not in globals() or "feature_names" not in globals():
        X, y, feature_names = make_planted_interaction_dataset(n_samples=2000, n_features=8)

    threshold_candidate = DiscreteFunctionCandidate(
        kind="discrete_function_soft_threshold",
        feature_indices=(3,),
        thresholds=(0.0,),
        direction="ge",
        scales=(float(np.std(X[:, 3])),),
        sharpness=12.0,
        mode="soft",
    )
    rectangle_candidate = DiscreteFunctionCandidate(
        kind="discrete_function_soft_rectangle",
        feature_indices=(3, 7),
        intervals=((-1.0, 1.0), (-1.0, 1.0)),
        scales=(float(np.std(X[:, 3])), float(np.std(X[:, 7]))),
        sharpness=12.0,
        mode="soft",
    )

    for candidate in [threshold_candidate, rectangle_candidate]:
        vector = evaluate_discrete_candidate(X, candidate)
        mask = evaluate_discrete_mask(X, candidate)
        print("\n", describe_discrete_candidate(candidate, feature_names), sep="")
        print("features:", discrete_feature_names(candidate, feature_names))
        print("candidate first 5:", np.asarray(vector[:5]))
        print("mask first 5:", np.asarray(mask[:5]))
except Exception as e:
    print("Discrete representation demo failed:", type(e).__name__, e)


## 40. Split-aware discrete ranking and CUDA selector scores

`EngineConfig.metric_names` still controls the metrics reported on final interactions. Discrete candidate ordering is separate and defaults to `discrete_ranking='split_aware'` so threshold/interval/rectangle candidates are not selected by Pearson alone.

Split-aware selector components:

- `mutual_info`: binned mutual information between split mask and target
- `variance_reduction`: soft impurity/variance reduction
- `residual_abs_corr`: absolute correlation against baseline residuals
- `residual_r2_gain`: residual R2-style gain

CUDA exposes the same selector API through `backend.score_discrete_selection_candidates(...)` for soft candidates. CPU/NumPy use the Python reference path.


In [ ]:
try:
    from gafime import ComputeBudget, EngineConfig
    from gafime.discrete import (
        describe_discrete_candidate,
        rank_discrete_selection_scores,
        score_discrete_selection_candidates,
    )
    from gafime.planning.discrete import plan_discrete_candidates

    if "X" not in globals() or "y" not in globals() or "feature_names" not in globals():
        X, y, feature_names = make_planted_interaction_dataset(n_samples=3000, n_features=10)

    feature_scores = {}
    for j in range(X.shape[1]):
        corr = np.corrcoef(X[:, j], y)[0, 1]
        feature_scores[j] = abs(float(corr)) if np.isfinite(corr) else 0.0

    cfg = EngineConfig(
        enable_discrete_functions=True,
        discrete_mode="soft",
        discrete_ranking="split_aware",
        metric_names=("mutual_info", "r2"),
        budget=ComputeBudget(
            max_discrete_candidates=1_000,
            max_thresholds_per_feature=5,
            max_intervals_per_feature=4,
            max_feature_pairs_for_rectangles=10,
            top_k_features_for_discrete=6,
        ),
    )

    candidates, warnings = plan_discrete_candidates(X, feature_scores, cfg)
    print("planned candidates:", len(candidates), "warnings:", warnings[:3])

    baseline_pred = np.full_like(y, float(np.mean(y)))
    scores = score_discrete_selection_candidates(
        X,
        y,
        candidates[:200],
        baseline_pred=baseline_pred,
        mi_bins=cfg.mi_bins,
    )
    ranked = rank_discrete_selection_scores(scores)

    print("\nTop split-aware candidates:")
    for candidate, rank_score in sorted(ranked.items(), key=lambda item: item[1], reverse=True)[:5]:
        print(round(rank_score, 4), describe_discrete_candidate(candidate, feature_names), scores[candidate])
except Exception as e:
    print("Split-aware ranking demo failed:", type(e).__name__, e)


## 41. Backend, Rust helper, and release workflow notes

Backend behavior in v0.4.0:

- CUDA/GPU discrete feature engineering supports soft/vectorized mode only.
- GPU hard mode raises `GPU feature engineering with discrete hard mode is not supported!`.
- CPU/NumPy can evaluate hard mode with vectorized host comparisons.
- Metal soft mode is GPU-compatible when the native Metal backend is present; hard mode is not a branch-heavy GPU path.
- Candidate launch ordering is cache-local, but GAFIME does not reserve or pin CUDA L2 cache.

Rust helper import migration:

```python
from gafime import subfunctions
```

Use `subfunctions` in public examples instead of direct `import gafime_cpu`; the Rust extension name remains an implementation detail.

Release build notes:

- v0.4.0 targets Python 3.10-3.14 wheels.
- CUDA workflow uses CUDA Toolkit 13.2.
- GPU wheel policy targets NVIDIA Turing and newer architectures.
- Release publication is gated on the `v0.4.0` tag and maintainer-controlled PyPI credentials.


In [ ]:
try:
    import gafime
    from gafime import ComputeBudget, EngineConfig, subfunctions
    from gafime.backends import resolve_backend
    from gafime.discrete import GPU_HARD_MODE_ERROR

    print("gafime version:", getattr(gafime, "__version__", None))
    print("GPU hard-mode error:", GPU_HARD_MODE_ERROR)
    print("subfunctions module:", subfunctions)
    print("subfunctions OTSEncoder:", getattr(subfunctions, "OTSEncoder", None))
    print("subfunctions BatchScheduler:", getattr(subfunctions, "BatchScheduler", None))

    if "X" in globals() and "y" in globals():
        cfg = EngineConfig(enable_discrete_functions=True, discrete_mode="hard", backend="auto")
        backend, warnings = resolve_backend(cfg, X[:100], y[:100])
        print("resolved backend:", backend.info())
        print("warnings:", warnings[:3])
        print("hard mode is legal only when backend.info().is_gpu is False")

    print("\nWheel target summary: cp310, cp311, cp312, cp313, cp314; CUDA 13.2; Turing+")
except Exception as e:
    print("Backend/release notes demo failed:", type(e).__name__, e)


## Conclusion

GAFIME v0.4.0 expands the engine from continuous interaction mining into discrete function representations while preserving the existing `GafimeEngine`, `EngineConfig`, and `ComputeBudget` style. The release adds threshold, interval, value-gated, and rectangle families; split-aware ranking; native soft CUDA/Metal paths; cache-local Rust scheduling; and updated wheel/release workflows.

Bu notebook üç rol üstlenir:

1. **Kullanıcı kitabı:** GAFIME nasıl kullanılır?
2. **Developer reference:** Hangi API ne parametre alır?
3. **Agent skill substrate:** Codex/Claude/GPT gibi ajanlar hangi smoke testleri çalıştırmalı?

Yeni API eklendikçe buraya bir bölüm eklemek iyi release hijyeni olur.
